# 🎯 Session 10: GPU 클라우드 접속 & vLLM 서빙 환경 구축

## 📋 학습 목표

1️⃣ 주요 GPU 클라우드 서비스를 비교하고 선택 기준을 배운다  
2️⃣ vLLM의 구조와 장점을 이해한다  
3️⃣ vLLM을 설치하고 OpenAI 호환 API 서버를 구축한다  
4️⃣ 서빙 벤치마크를 수행하고 최적화 방법을 익힌다  

---

### 🖥️ 실습 환경
- **GPU**: 클라우드 GPU 권장 (A100, L4 등)
- **로컬**: 명령어/설정 학습 위주 (RTX 4060에서도 1.5B 모델은 가능)
- **패키지**: vllm, openai

> 📌 이 노트북은 명령어와 설정을 학습하는 것이 목적입니다.  
> 실제 클라우드 접속은 선택사항이며, 로컬에서도 개념을 익힐 수 있습니다.

## 1️⃣ GPU 클라우드 서비스 비교

로컬 GPU로 부족한 경우, 클라우드 GPU를 활용할 수 있습니다.

### 📊 주요 클라우드 GPU 서비스

| 서비스 | GPU | 무료 제공 | 가격대 | 특징 |
|--------|-----|----------|--------|------|
| **Google Colab** | T4 (무료) / A100 (Pro) | ✅ T4 15GB | $10/월 (Pro) | 주피터 환경, 구글 드라이브 연동 |
| **Kaggle** | T4 x2 / P100 | ✅ 30시간/주 | 무료 | 데이터셋 풍부, 경진대회 |
| **RunPod** | A100 / H100 | ❌ | $0.44/시간~ | 다양한 GPU, 템플릿 제공 |
| **Lambda** | A100 / H100 | ❌ | $0.50/시간~ | ML 특화, 빠른 셋업 |
| **Vast.ai** | 다양 | ❌ | $0.10/시간~ | 저렴, 커뮤니티 GPU |

In [ ]:
# 📊 GPU 클라우드 비용 계산기
cloud_gpus = [
    {"name": "Colab Free (T4)",      "vram": 15,  "price_hr": 0,     "monthly": 0},
    {"name": "Colab Pro (A100)",      "vram": 40,  "price_hr": 0.12,  "monthly": 10},
    {"name": "Kaggle (T4 x2)",       "vram": 30,  "price_hr": 0,     "monthly": 0},
    {"name": "RunPod (A100 40GB)",    "vram": 40,  "price_hr": 0.44,  "monthly": None},
    {"name": "RunPod (A100 80GB)",    "vram": 80,  "price_hr": 0.79,  "monthly": None},
    {"name": "RunPod (H100 80GB)",    "vram": 80,  "price_hr": 2.49,  "monthly": None},
    {"name": "Vast.ai (RTX 3090)",   "vram": 24,  "price_hr": 0.15,  "monthly": None},
    {"name": "Vast.ai (A100 40GB)",  "vram": 40,  "price_hr": 0.35,  "monthly": None},
]

print("📊 GPU 클라우드 비용 비교")
print("=" * 70)
print(f"{'서비스':<25} {'VRAM':>6} {'시간당':>10} {'8시간/일×30일':>15}")
print("-" * 70)

for gpu in cloud_gpus:
    monthly_cost = gpu['monthly'] if gpu['monthly'] is not None else gpu['price_hr'] * 8 * 30
    price_str = f"${gpu['price_hr']:.2f}" if gpu['price_hr'] > 0 else "무료"
    monthly_str = f"${monthly_cost:.0f}" if monthly_cost > 0 else "무료"
    print(f"{gpu['name']:<25} {gpu['vram']:>4}GB {price_str:>10} {monthly_str:>15}")

print(f"\n💡 추천:")
print(f"  🔹 학습/실습: Colab Free 또는 Kaggle (무료)")
print(f"  🔹 파인튜닝: RunPod A100 또는 Vast.ai")
print(f"  🔹 서빙: RunPod (안정적), Vast.ai (저렴)")

## 2️⃣ 클라우드 접속 및 환경 설정

각 클라우드 서비스별 접속 및 환경 설정 방법을 알아봅니다.

In [ ]:
# 📋 Google Colab 환경 설정 가이드
colab_setup = """
🔧 Google Colab 설정 가이드
══════════════════════════════════════════

1. https://colab.research.google.com 접속
2. 새 노트북 생성
3. 런타임 → 런타임 유형 변경 → GPU 선택
4. 아래 코드로 GPU 확인:

   !nvidia-smi
   import torch
   print(torch.cuda.get_device_name(0))

5. 필수 패키지 설치:
   !pip install vllm transformers accelerate

💡 Colab Pro 팁:
   - A100 사용 시 런타임 유형에서 'A100' 직접 선택
   - 고용량 RAM 옵션 활성화 가능
   - 세션 최대 24시간 유지
"""
print(colab_setup)

In [ ]:
# 📋 RunPod 환경 설정 가이드
runpod_setup = """
🔧 RunPod 설정 가이드
══════════════════════════════════════════

1. https://runpod.io 회원가입 및 결제 설정

2. GPU Pod 생성:
   - Templates → 'PyTorch' 또는 'vLLM' 선택
   - GPU: A100 40GB 또는 80GB 선택
   - Container Disk: 50GB 이상
   - Volume Disk: 100GB (모델 저장용)

3. SSH 접속:
   ssh -p PORT root@IP -i ~/.ssh/id_rsa

4. 또는 웹 터미널/Jupyter 사용

5. 환경 설정:
   pip install vllm transformers
   huggingface-cli login

💡 RunPod 팁:
   - Spot Instance로 최대 80% 할인
   - Volume을 사용하면 모델을 재다운로드할 필요 없음
   - 사용하지 않을 때는 반드시 Pod 정지!
"""
print(runpod_setup)

## 3️⃣ vLLM 소개

**vLLM**은 대규모 언어 모델을 빠르게 서빙하기 위한 오픈소스 엔진입니다.

### 🚀 vLLM의 핵심 기술

- 🔹 **PagedAttention**: GPU 메모리를 효율적으로 관리하여 처리량 증가
- 🔹 **Continuous Batching**: 동적 배칭으로 대기 시간 최소화
- 🔹 **KV Cache 최적화**: 키-밸류 캐시를 효율적으로 관리
- 🔹 **OpenAI 호환 API**: 기존 코드 그대로 사용 가능

### 📊 vLLM vs 다른 서빙 프레임워크

| 기능 | vLLM | TGI | Ollama | Transformers |
|------|------|-----|--------|-------------|
| 처리량 | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | ⭐ |
| 배칭 | 연속 배칭 | 연속 배칭 | 단일 | 없음 |
| API | OpenAI 호환 | 자체 API | 자체+OpenAI | 없음 |
| 설치 난이도 | 보통 | 쉬움 | 매우 쉬움 | 매우 쉬움 |
| GPU 요구 | 높음 | 높음 | 낮음 | 낮음 |

In [ ]:
# 📊 vLLM vs Ollama vs Transformers 서빙 비교
print("📊 LLM 서빙 프레임워크 비교")
print("=" * 65)

comparisons = [
    ("사용 난이도",   "보통",     "매우 쉬움",  "매우 쉬움"),
    ("처리량(QPS)",  "높음 (10x)", "보통 (3x)",  "기준 (1x)"),
    ("동시 요청",    "수백 개",   "수십 개",    "1개"),
    ("배칭",         "연속 배칭",  "단일 배칭",  "없음"),
    ("API 호환",     "OpenAI",   "OpenAI",    "없음"),
    ("최소 VRAM",    "16GB+",    "4GB+",      "2GB+"),
    ("주요 용도",    "프로덕션",  "개발/테스트", "실험/학습"),
]

print(f"{'항목':<15} {'vLLM':>15} {'Ollama':>15} {'Transformers':>15}")
print("-" * 65)
for item, vllm, ollama, tf in comparisons:
    print(f"{item:<15} {vllm:>15} {ollama:>15} {tf:>15}")

print(f"\n💡 선택 가이드:")
print(f"  🔹 프로덕션 서빙 → vLLM")
print(f"  🔹 로컬 개발/테스트 → Ollama")
print(f"  🔹 연구/실험 → Transformers")

## 4️⃣ vLLM 설치 및 기본 사용법

vLLM을 설치하고 기본적인 텍스트 생성을 수행합니다.

In [ ]:
# 📦 vLLM 설치 명령어
print("📦 vLLM 설치 방법")
print("=" * 50)
print()
print("# 기본 설치")
print("pip install vllm")
print()
print("# CUDA 버전에 맞는 설치 (CUDA 12.1 기준)")
print("pip install vllm --extra-index-url https://download.pytorch.org/whl/cu121")
print()
print("# 최신 개발 버전")
print("pip install vllm --pre")
print()
print("⚠️ 주의사항:")
print("  - Python 3.9+ 필요")
print("  - CUDA 12.1+ 권장")
print("  - GPU VRAM 16GB+ 권장 (7B 모델 기준)")
print("  - RTX 4060 (8GB)에서는 1.5B 모델만 가능")

In [ ]:
# 🔧 vLLM 오프라인 추론 (Python API)
# ⚠️ 충분한 GPU 메모리가 있는 환경에서 실행하세요

print("🔧 vLLM 오프라인 추론 예시 코드")
print("=" * 50)

vllm_offline_code = '''
from vllm import LLM, SamplingParams

# 모델 로딩
llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    dtype="float16",           # FP16 정밀도
    max_model_len=2048,        # 최대 시퀀스 길이
    gpu_memory_utilization=0.8, # GPU 메모리 사용률
)

# 샘플링 파라미터
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=256,
)

# 프롬프트 정의
prompts = [
    "인공지능의 미래에 대해 설명해주세요.",
    "Python의 장점을 3가지 알려주세요.",
    "한국의 전통 음식 중 대표적인 것은?",
]

# 추론 (배칭 자동 처리)
outputs = llm.generate(prompts, sampling_params)

# 결과 출력
for output in outputs:
    prompt = output.prompt
    generated = output.outputs[0].text
    print(f"Prompt: {prompt[:50]}...")
    print(f"Output: {generated[:200]}...")
    print()
'''

print(vllm_offline_code)

In [ ]:
# 🚀 vLLM 실제 실행 (GPU 메모리 충분한 경우)
import torch

try:
    from vllm import LLM, SamplingParams
    vllm_available = True
    print("✅ vLLM이 설치되어 있습니다.")
except ImportError:
    vllm_available = False
    print("⚠️ vLLM이 설치되어 있지 않습니다.")
    print("   pip install vllm")

if vllm_available and torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"\n📊 사용 가능한 VRAM: {vram:.1f}GB")
    
    if vram >= 6:  # 최소 6GB 필요 (1.5B 모델)
        print("\n🚀 1.5B 모델로 vLLM 추론 시작...")
        
        llm = LLM(
            model="Qwen/Qwen2.5-1.5B-Instruct",
            dtype="float16",
            max_model_len=1024,
            gpu_memory_utilization=0.7,
        )
        
        sampling_params = SamplingParams(
            temperature=0.7,
            top_p=0.9,
            max_tokens=128,
        )
        
        prompts = [
            "파이썬에서 리스트와 딕셔너리의 차이점은?",
            "머신러닝과 딥러닝의 차이를 간단히 설명하면?",
        ]
        
        import time
        start = time.time()
        outputs = llm.generate(prompts, sampling_params)
        elapsed = time.time() - start
        
        total_tokens = 0
        for output in outputs:
            generated = output.outputs[0].text
            total_tokens += len(output.outputs[0].token_ids)
            print(f"\n❓ {output.prompt[:50]}...")
            print(f"💬 {generated[:200]}")
        
        print(f"\n📊 성능: {total_tokens}개 토큰 / {elapsed:.2f}초 = {total_tokens/elapsed:.1f} tok/s")
        
        # 메모리 해제
        del llm
        import gc
        gc.collect()
        torch.cuda.empty_cache()
    else:
        print(f"⚠️ VRAM이 부족합니다. 클라우드 환경에서 실행해주세요.")
else:
    print("\n📌 vLLM 실행은 클라우드 환경에서 진행해주세요.")

## 5️⃣ vLLM OpenAI 호환 API 서버

vLLM은 OpenAI API와 호환되는 HTTP 서버를 제공합니다.

In [ ]:
# 🌐 vLLM API 서버 시작 명령어
print("🌐 vLLM OpenAI 호환 API 서버")
print("=" * 60)

server_commands = {
    "기본 실행": """
python -m vllm.entrypoints.openai.api_server \\
    --model Qwen/Qwen2.5-7B-Instruct \\
    --port 8000""",
    
    "최적화 옵션": """
python -m vllm.entrypoints.openai.api_server \\
    --model Qwen/Qwen2.5-7B-Instruct \\
    --port 8000 \\
    --dtype float16 \\
    --max-model-len 4096 \\
    --gpu-memory-utilization 0.85 \\
    --max-num-batched-tokens 8192 \\
    --tensor-parallel-size 1""",
    
    "양자화 모델": """
python -m vllm.entrypoints.openai.api_server \\
    --model Qwen/Qwen2.5-7B-Instruct-AWQ \\
    --quantization awq \\
    --port 8000""",
    
    "멀티 GPU (2장)": """
python -m vllm.entrypoints.openai.api_server \\
    --model Qwen/Qwen2.5-14B-Instruct \\
    --tensor-parallel-size 2 \\
    --port 8000""",
}

for name, cmd in server_commands.items():
    print(f"\n🔹 {name}:")
    print(cmd)
    print()

In [ ]:
# 📡 vLLM API 서버에 요청 보내기
# (서버가 실행 중일 때 사용)

print("📡 vLLM API 서버 사용법 (OpenAI 호환)")
print("=" * 55)

api_usage_code = '''
from openai import OpenAI

# vLLM 서버에 연결
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="token-not-needed",  # vLLM은 API 키 불필요
)

# Chat Completions API
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[
        {"role": "system", "content": "당신은 유능한 AI 어시스턴트입니다."},
        {"role": "user", "content": "파이썬의 장점을 알려주세요."},
    ],
    temperature=0.7,
    max_tokens=256,
)

print(response.choices[0].message.content)

# 스트리밍
stream = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[{"role": "user", "content": "Hello!"}],
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
'''

print(api_usage_code)

In [ ]:
# 🔧 curl로 테스트하는 방법
print("🔧 curl로 vLLM API 테스트")
print("=" * 60)

curl_commands = [
    ("모델 목록 확인", 'curl http://localhost:8000/v1/models'),
    ("Chat Completion", '''curl http://localhost:8000/v1/chat/completions \\
  -H "Content-Type: application/json" \\
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "messages": [{"role": "user", "content": "Hello!"}],
    "max_tokens": 100
  }' '''),
    ("서버 상태 확인", 'curl http://localhost:8000/health'),
]

for name, cmd in curl_commands:
    print(f"\n🔹 {name}:")
    print(f"  {cmd}")

## 6️⃣ vLLM 주요 설정 파라미터

성능 최적화를 위한 주요 설정들을 알아봅니다.

In [ ]:
# ⚙️ vLLM 주요 파라미터 정리
params = [
    ("--model", "HuggingFace 모델 이름", "필수"),
    ("--dtype", "데이터 타입 (auto/float16/bfloat16)", "auto"),
    ("--max-model-len", "최대 시퀀스 길이", "모델 기본값"),
    ("--gpu-memory-utilization", "GPU 메모리 사용 비율", "0.9"),
    ("--tensor-parallel-size", "텐서 병렬 GPU 수", "1"),
    ("--max-num-batched-tokens", "배치당 최대 토큰 수", "자동"),
    ("--max-num-seqs", "동시 처리 최대 시퀀스 수", "256"),
    ("--quantization", "양자화 방법 (awq/gptq/squeezellm)", "없음"),
    ("--port", "API 서버 포트", "8000"),
    ("--host", "바인딩 호스트", "0.0.0.0"),
]

print("⚙️ vLLM 주요 서버 파라미터")
print("=" * 70)
print(f"{'파라미터':<32} {'설명':<25} {'기본값':>10}")
print("-" * 70)
for param, desc, default in params:
    print(f"{param:<32} {desc:<25} {default:>10}")

print(f"\n💡 RTX 4060 (8GB) 최적 설정:")
print(f"  --dtype float16")
print(f"  --max-model-len 1024")
print(f"  --gpu-memory-utilization 0.8")
print(f"  --model Qwen/Qwen2.5-1.5B-Instruct")

## 7️⃣ 서빙 벤치마크

vLLM 서버의 성능을 측정하는 방법을 알아봅니다.

In [ ]:
# 📊 벤치마크 스크립트 (서버 실행 중일 때)
print("📊 vLLM 벤치마크 스크립트")
print("=" * 55)

benchmark_code = '''
import time
import concurrent.futures
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="not-needed",
)

def single_request(prompt):
    """단일 요청을 보내고 시간을 측정합니다."""
    start = time.time()
    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-7B-Instruct",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=128,
        temperature=0.7,
    )
    elapsed = time.time() - start
    tokens = response.usage.completion_tokens
    return elapsed, tokens

# 동시 요청 벤치마크
prompts = ["파이썬의 장점을 설명해주세요."] * 10

print("🚀 동시 요청 벤치마크 시작...")
start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    results = list(executor.map(single_request, prompts))

total_time = time.time() - start
total_tokens = sum(r[1] for r in results)

print(f"📊 결과:")
print(f"  총 요청: {len(prompts)}개")
print(f"  총 시간: {total_time:.2f}초")
print(f"  QPS: {len(prompts)/total_time:.2f} req/sec")
print(f"  총 토큰: {total_tokens}")
print(f"  처리량: {total_tokens/total_time:.1f} tok/sec")
'''

print(benchmark_code)

In [ ]:
# 📊 예상 벤치마크 결과 (참고용)
print("📊 GPU별 예상 vLLM 성능 (Qwen2.5-7B, FP16)")
print("=" * 65)
print(f"{'GPU':<20} {'VRAM':>6} {'단일 속도':>12} {'동시10 QPS':>12} {'비고':>12}")
print("-" * 65)

benchmarks = [
    ("RTX 4060",    "8GB",   "불가",        "불가",      "1.5B만 가능"),
    ("RTX 4090",    "24GB",  "~40 tok/s",  "~3 QPS",   "가능"),
    ("A100 40GB",   "40GB",  "~80 tok/s",  "~8 QPS",   "권장"),
    ("A100 80GB",   "80GB",  "~90 tok/s",  "~12 QPS",  "최적"),
    ("H100 80GB",   "80GB",  "~150 tok/s", "~20 QPS",  "최고성능"),
]

for gpu, vram, single, concurrent, note in benchmarks:
    print(f"{gpu:<20} {vram:>6} {single:>12} {concurrent:>12} {note:>12}")

print(f"\n📌 참고: 실제 성능은 모델, 입력 길이, 배치 크기 등에 따라 다릅니다.")

## 8️⃣ Ollama vs vLLM 서빙 실습

같은 모델을 Ollama와 vLLM으로 서빙했을 때의 차이를 알아봅니다.

In [ ]:
# 🔄 Ollama vs vLLM 비교 요약
print("🔄 Ollama vs vLLM 비교 요약")
print("=" * 65)

comparison = [
    ("설치 난이도",     "한 줄 설치",        "pip install vllm"),
    ("모델 포맷",       "GGUF (양자화)",     "HF (FP16/양자화)"),
    ("최소 VRAM",       "4GB",               "16GB+ (7B 기준)"),
    ("배칭",            "단일 요청",          "연속 배칭"),
    ("동시 처리",       "제한적",            "수백 요청"),
    ("처리량",          "보통",              "매우 높음"),
    ("API",             "자체 + OpenAI",     "OpenAI 호환"),
    ("권장 GPU",        "소비자급",          "서버급 (A100+)"),
    ("적합한 용도",     "개발/테스트",       "프로덕션 서빙"),
]

print(f"{'항목':<18} {'Ollama':>20} {'vLLM':>22}")
print("-" * 65)
for item, ollama, vllm in comparison:
    print(f"{item:<18} {ollama:>20} {vllm:>22}")

print(f"\n💡 결론:")
print(f"  🔹 개발/실습 → Ollama (쉽고 가벼움)")
print(f"  🔹 프로덕션/다수 사용자 → vLLM (높은 처리량)")
print(f"  🔹 RTX 4060 → Ollama 사용 (VRAM 제한)")

In [ ]:
# 📌 실습 정리
print("📌 오늘의 핵심 정리")
print("=" * 50)
print("  1️⃣ GPU 클라우드 서비스 선택 시 비용, VRAM, 안정성 고려")
print("  2️⃣ vLLM은 PagedAttention으로 높은 처리량 달성")
print("  3️⃣ OpenAI 호환 API로 기존 코드 재사용 가능")
print("  4️⃣ 개발은 Ollama, 프로덕션은 vLLM이 적합")
print("  5️⃣ GPU 메모리에 맞는 모델 크기와 양자화 선택이 중요")

---

## 🎯 실습 과제

1️⃣ Google Colab에 접속하여 GPU 환경을 확인해보세요  
2️⃣ Colab에서 vLLM을 설치하고 1.5B 모델로 추론해보세요  
3️⃣ Ollama와 vLLM의 응답 시간을 같은 질문으로 비교해보세요  

---

## 📚 참고 자료
- [vLLM 공식 문서](https://docs.vllm.ai/)
- [vLLM GitHub](https://github.com/vllm-project/vllm)
- [RunPod 문서](https://docs.runpod.io/)
- [Google Colab](https://colab.research.google.com)